# Causal Trace Recipe on a Tiny Transformer

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Causal tracing asks: if a corrupted prompt changes the answer, which saved internal states restore the clean answer when patched back in? In a production transformer you usually sweep residual-stream, MLP, or attention sites by layer and position. Here we keep the same structure but use a tiny local token model.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1601
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyBlock(nn.Module):
    """One residual MLP block used as a transformer-sized stand-in."""

    def __init__(self, width: int) -> None:
        """Initialize the block."""
        super().__init__()
        self.ln = nn.LayerNorm(width)
        self.up = nn.Linear(width, width)
        self.down = nn.Linear(width, width)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply layer norm, MLP, and residual addition."""
        hidden = torch.relu(self.up(self.ln(x)))
        return x + self.down(hidden)


class TinyTransformer(nn.Module):
    """Small token model with named blocks and sequence positions."""

    def __init__(self, vocab_size: int = 16, width: int = 12, depth: int = 3) -> None:
        """Initialize embeddings, blocks, and unembedding head."""
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList([TinyBlock(width) for _ in range(depth)])
        self.unembed = nn.Linear(width, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Return logits for every token position."""
        x = self.embed(tokens)
        for block in self.blocks:
            x = block(x)
        return self.unembed(x)

In [ ]:
model = TinyTransformer().eval()
clean_tokens = torch.tensor([[1, 2, 3, 4]])
corrupt_tokens = torch.tensor([[1, 2, 9, 4]])
clean = tl.log_forward_pass(
    model, clean_tokens, vis_opt="none", intervention_ready=True, name="clean"
)
corrupt = tl.log_forward_pass(
    model, corrupt_tokens, vis_opt="none", intervention_ready=True, name="corrupt"
)

clean_logits = clean.layer_list[-1].activation
corrupt_logits = corrupt.layer_list[-1].activation
target_token = int(clean_logits[0, -1].argmax())
base_gap = float(clean_logits[0, -1, target_token] - corrupt_logits[0, -1, target_token])
print(f"clean target token: {target_token}")
print(f"clean-minus-corrupt logit gap: {base_gap:.4f}")

We sweep ReLU MLP outputs because they are easy to interpret in this toy model and map cleanly to common transformer MLP-site workflows. `find_sites` gives concrete labels, while `tl.label(...)` makes each replay target explicit and portable inside the current log.


In [ ]:
relu_sites = clean.find_sites(tl.func("relu"))
site_labels = [site.layer_label for site in relu_sites]
print(site_labels)

In [ ]:
scores: list[float] = []
for label in site_labels:
    clean_activation = clean[label].activation
    patched = corrupt.fork(f"patch_{label}")
    patched.set(tl.label(label), clean_activation, confirm_mutation=True)
    patched.replay()
    patched_logits = patched.layer_list[-1].activation
    restored = float(patched_logits[0, -1, target_token] - corrupt_logits[0, -1, target_token])
    scores.append(restored / base_gap if abs(base_gap) > 1e-6 else 0.0)

for label, score in zip(site_labels, scores):
    print(f"{label:18s} restored {score: .3f} of the clean-corrupt gap")

A heatmap is useful once the sweep has more than a handful of rows or columns. TorchLens provides the rendering helper; the scoring convention is still yours, so label axes and sign meaning explicitly in real analyses.


In [ ]:
score_tensor = torch.tensor(scores, dtype=torch.float32).unsqueeze(1)
ax = tl.viz.causal_trace_heatmap(score_tensor, signs="all", cmap="viridis")
ax.set_yticks(range(len(site_labels)))
ax.set_yticklabels(site_labels)
ax.set_xticks([0])
ax.set_xticklabels(["final token"])
ax.set_title("Clean activation patch restores target logit")
plt.tight_layout()
plt.show()